# CM3065 / Intelligent Signal Processing — Exercise 4 (DeepSpeech ASR + WER)

This notebook implements **Exercise 4.1** using the **exact DeepSpeech models specified in `Ex4_models.pdf`** (English v0.9.3 + Scribosermo community models for Spanish/Italian), and evaluates outputs against the provided reference transcriptions using **Word Error Rate (WER)**.

> **Important note (execution environment):**  
> DeepSpeech v0.9.x is not available on Apple Silicon (arm64) / Python 3.9+ natively.So i used Docker and saved the predictions to `ex4_predictions.csv`.  
> This notebook can then computes WER and generates the tables/analysis for excercise 4.




In [2]:
##importing libraries
import os
import re
import wave
import numpy as np
import pandas as pd

print("Notebook ready.")

Notebook ready.


In [3]:
BASE_DIR = os.getcwd()  # default: current folder

LANG_DIRS = {
    "EN": os.path.join(BASE_DIR, "EN"),
    "ES": os.path.join(BASE_DIR, "ES"),
    "IT": os.path.join(BASE_DIR, "IT"),
}

MODEL_FILES = {
    "EN": {"model": "deepspeech-0.9.3-models.pbmm", "scorer": "deepspeech-0.9.3-models.scorer"},
    "ES": {"model": "model.pbmm", "scorer": "kenlm.scorer"},
    "IT": {"model": "model.pbmm", "scorer": "kenlm.scorer"},
}

PRED_CSV = os.path.join(BASE_DIR, "ex4_predictions.csv")

# Defines the base working directory used to locate language folders and output files.
# The default uses the current working directory to keep the script portable across environments.
BASE_DIR = os.getcwd()  # default: current folder

# Builds absolute paths to each language folder so audio and model assets can be located consistently.
# Keys match the dataset’s language folder naming convention.
LANG_DIRS = {
    "EN": os.path.join(BASE_DIR, "EN"),
    "ES": os.path.join(BASE_DIR, "ES"),
    "IT": os.path.join(BASE_DIR, "IT"),
}

# Specifies the expected model and scorer filenames for each language setup.
# These names are used later to construct full paths and to validate that required files exist.
MODEL_FILES = {
    "EN": {"model": "deepspeech-0.9.3-models.pbmm", "scorer": "deepspeech-0.9.3-models.scorer"},
    "ES": {"model": "model.pbmm", "scorer": "kenlm.scorer"},
    "IT": {"model": "model.pbmm", "scorer": "kenlm.scorer"},
}

# Defines where the combined prediction output CSV is expected to be found or written.
# The file stores per-audio-file transcription outputs for later WER evaluation and reporting.
PRED_CSV = os.path.join(BASE_DIR, "ex4_predictions.csv")

# Prints the resolved language directory paths so it is clear where the script will look for each dataset.
for k, v in LANG_DIRS.items():
    print(k, "->", v)


EN -> /Users/paris/Desktop/isp/Exercise4/deepspeech/EN
ES -> /Users/paris/Desktop/isp/Exercise4/deepspeech/ES
IT -> /Users/paris/Desktop/isp/Exercise4/deepspeech/IT


## Reference transcriptions (from `Ex4_audio_files_transcriptions.pdf`)

These are the official reference strings used to compute WER.


In [5]:
REF = {
    # English
    "checkin.wav": "Where is the check-in desk?",
    "checkin_child.wav": "Where is the check-in desk?",
    "parents.wav": "I have lost my parents.",
    "parents_child.wav": "I have lost my parents.",
    "suitcase.wav": "Please, I have lost my suitcase.",
    "suitcase_child.wav": "Please, I have lost my suitcase.",
    "what_time.wav": "What time is my plane?",
    "what_time_child.wav": "What time is my plane?",
    "your_sentence1.wav": "I don't want anything to drink i'm rather hungry.",
    "your_sentence2.wav": "I excercise for an hour every morning.",

    # Italian
    "checkin_it.wav": "Dove e' il bancone?",
    "parents_it.wav": "Ho perso i miei genitori.",
    "suitcase_it.wav": "Per favore, ho perso la mia valigia.",
    "what_time_it.wav": "A che ora e’ il mio aereo?",
    "where_it.wav": "Dove sono i ristoranti e i negozi?",

    # Spanish
    "checkin_es.wav": "¿Dónde están los mostradores?",
    "parents_es.wav": "He perdido a mis padres.",
    "suitcase_es.wav": "Por favor, he perdido mi maleta.",
    "what_time_es.wav": "¿A qué hora es mi avión?",
    "where_es.wav": "¿Dónde están los restaurantes y las tiendas?",
}

len(REF), list(REF.items())[:3]

(20,
 [('checkin.wav', 'Where is the check-in desk?'),
  ('checkin_child.wav', 'Where is the check-in desk?'),
  ('parents.wav', 'I have lost my parents.')])

## Text normalization + WER implementation

WER is computed with Levenshtein distance over **word tokens** after normalization.
Normalization removes punctuation, normalizes whitespace, and lowercases while keeping accented characters.


In [7]:
# Normalizes transcript text into a consistent form for word-level comparison.
# The transformation reduces mismatches caused by punctuation, casing, and inconsistent spacing.
def normalize_text(s: str) -> str:
    # Converts missing inputs to an empty string, lowercases, and trims surrounding whitespace.
    s = (s or "").lower().strip()

    # Replaces non-supported characters with spaces while keeping:
    # - basic latin letters and digits
    # - common accented latin characters used in ES/IT
    # - apostrophes and hyphens
    # This helps keep words comparable across different transcription outputs.
    s = re.sub(r"[^a-z0-9àèéìòóùüñç'’\- ]+", " ", s)

    # Normalizes curly apostrophes into straight apostrophes and converts hyphens into spaces.
    # Hyphen splitting avoids treating compound words as a single token when transcriptions differ.
    s = s.replace("’", "'").replace("-", " ")

    # Collapses repeated whitespace into a single space and trims again for stable tokenization.
    s = re.sub(r"\s+", " ", s).strip()
    return s

# Computes Word Error Rate (WER) as a fraction using word-level edit distance.
# WER = (minimum edits to transform reference words into hypothesis words) / (number of reference words).
def wer(ref: str, hyp: str) -> float:
    # Splits normalized reference into tokens (words).
    ref_words = normalize_text(ref).split()

    # Splits normalized hypothesis into tokens (words).
    hyp_words = normalize_text(hyp).split()

    # n and m represent the lengths of the reference and hypothesis token sequences.
    n = len(ref_words)
    m = len(hyp_words)

    # dp[i, j] stores the minimum edit operations required to align:
    # - the first i reference words
    # - with the first j hypothesis words
    dp = np.zeros((n + 1, m + 1), dtype=int)

    # Initializes base cases:
    # Converting i reference words to an empty hypothesis requires i deletions.
    for i in range(n + 1):
        dp[i, 0] = i

    # Converting an empty reference to j hypothesis words requires j insertions.
    for j in range(m + 1):
        dp[0, j] = j

    # Fills the dynamic programming table using standard Levenshtein distance recurrence.
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            # Substitution cost is 0 if words match, otherwise 1.
            cost = 0 if ref_words[i - 1] == hyp_words[j - 1] else 1

            # Chooses the lowest-cost operation among:
            # - deletion (drop a reference word)
            # - insertion (add a hypothesis word)
            # - substitution (replace reference word with hypothesis word)
            dp[i, j] = min(
                dp[i - 1, j] + 1,      # deletion
                dp[i, j - 1] + 1,      # insertion
                dp[i - 1, j - 1] + cost # substitution
            )

    # Returns WER as a fraction of reference length, guarding against divide-by-zero for empty references.
    return dp[n, m] / max(1, n)

print("WER sanity:", wer("a b c", "a b c"), wer("a b c", "a c"))


WER sanity: 0.0 0.3333333333333333


 ## Running DeepSpeech inference in this notebook


In [9]:
# Reads a WAV file and returns a mono int16 waveform, enforcing the exact format expected by DeepSpeech:
# - 16-bit PCM samples
# - 1 channel (mono) after conversion if needed
# - 16 kHz sample rate
def read_wav_16k_mono(path: str) -> np.ndarray:
    # Opens the WAV container and extracts core audio properties and raw frame bytes.
    with wave.open(path, "rb") as wf:
        sr = wf.getframerate()
        ch = wf.getnchannels()
        sw = wf.getsampwidth()

        # Reads all frames and interprets them as int16 PCM samples.
        audio = np.frombuffer(wf.readframes(wf.getnframes()), dtype=np.int16)

    # Validates sample width: DeepSpeech input is expected to be 16-bit PCM (2 bytes per sample).
    if sw != 2:
        raise ValueError(f"{path}: expected 16-bit PCM wav (sampwidth=2), got {sw}")

    # Converts stereo to mono by averaging channels, preserving int16 output type.
    if ch == 2:
        audio = audio.reshape(-1, 2).mean(axis=1).astype(np.int16)
    # Rejects multi-channel formats beyond mono/stereo to avoid ambiguous conversions.
    elif ch != 1:
        raise ValueError(f"{path}: unsupported channels={ch}")

    # Enforces 16 kHz sample rate to match DeepSpeech model requirements and avoid incorrect decoding.
    if sr != 16000:
        raise ValueError(f"{path}: expected 16kHz audio, got {sr}Hz")

    # Returns the validated, standardized mono waveform as int16 samples.
    return audio

# Runs DeepSpeech inference across all language folders and saves predictions into a CSV.
# This function is intended for environments where DeepSpeech is supported and loadable.
# When DeepSpeech is unavailable (common on incompatible architectures), a clear error is raised.
def try_run_inference(save_csv: str = PRED_CSV) -> pd.DataFrame:
    # Imports DeepSpeech only when inference is requested, allowing WER-only workflows to run without it.
    try:
        from deepspeech import Model
    except Exception as e:
        # Raises an environment-specific message explaining how to proceed when DeepSpeech cannot be imported.
        raise RuntimeError(
            "DeepSpeech is not available in this environment. "
            "Run inference in a compatible environment (e.g., Docker x86_64 + Python 3.8) "
            "and place ex4_predictions.csv next to this notebook."
        ) from e

    # Loads the language-specific DeepSpeech models and scorers into a dictionary keyed by language folder name.
    ds = {}
    for lang in ["EN", "ES", "IT"]:
        # Constructs absolute paths to the model and scorer files for the current language.
        model_path = os.path.join(LANG_DIRS[lang], MODEL_FILES[lang]["model"])
        scorer_path = os.path.join(LANG_DIRS[lang], MODEL_FILES[lang]["scorer"])

        # Validates required model assets exist before attempting to load them.
        if not os.path.exists(model_path):
            raise FileNotFoundError(f"Missing {lang} model file: {model_path}")
        if not os.path.exists(scorer_path):
            raise FileNotFoundError(f"Missing {lang} scorer file: {scorer_path}")

        # Initializes the DeepSpeech model and attaches the external scorer (language model).
        m = Model(model_path)
        m.enableExternalScorer(scorer_path)

        # Stores the ready-to-use model instance for later transcription.
        ds[lang] = m

    # Collects one transcription row per WAV file across all language folders.
    rows = []
    for lang in ["EN", "ES", "IT"]:
        # Lists WAV files from the language folder and sorts for stable, reproducible processing order.
        wavs = sorted([f for f in os.listdir(LANG_DIRS[lang]) if f.lower().endswith(".wav")])

        # Runs speech-to-text on each WAV file, enforcing format compliance via read_wav_16k_mono.
        for w in wavs:
            audio = read_wav_16k_mono(os.path.join(LANG_DIRS[lang], w))
            hyp = ds[lang].stt(audio)
            rows.append({"language": lang, "file": w, "prediction": hyp})

    # Builds a DataFrame containing all predictions and sorts results for consistent reporting.
    df_pred = pd.DataFrame(rows).sort_values(["language", "file"]).reset_index(drop=True)

    # Saves predictions to CSV so WER evaluation can run later without repeating model inference.
    df_pred.to_csv(save_csv, index=False)
    print("Saved predictions ->", save_csv)
    return df_pred

# df_pred = try_run_inference()


##  Load predictions

Load `ex4_predictions.csv` created by inference.  



In [11]:
PRED_CSV = "ex4_predictions.csv"

ALT = "ex4_predictions_with_wer.csv"   # change if your filename is slightly different

# Defines the expected prediction CSV filename that downstream notebook cells are designed to load.
PRED_CSV = "ex4_predictions.csv"

# Defines an alternate filename that may exist when predictions were saved with additional evaluation columns.
# This path is used as a fallback to rebuild the expected file format when PRED_CSV is missing.
ALT = "ex4_predictions_with_wer.csv"   # change if your filename is slightly different

# Checks whether the expected predictions file exists in the current working directory.
# If it does not exist, a conversion step attempts to create it from the alternate CSV.
if not os.path.exists(PRED_CSV):
    # Ensures the alternate CSV exists before attempting conversion.
    if not os.path.exists(ALT):
        raise FileNotFoundError(f"Neither {PRED_CSV} nor {ALT} exists in this folder.")

    # Loads the alternate CSV into a DataFrame so columns can be inspected and standardized.
    df = pd.read_csv(ALT)

    # Normalizes prediction column naming differences across CSV exports.
    # Some files label recognizer output as "hypothesis" or "HYP", while the notebook expects "prediction".
    if "hypothesis" in df.columns and "prediction" not in df.columns:
        df = df.rename(columns={"hypothesis": "prediction"})
    if "HYP" in df.columns and "prediction" not in df.columns:
        df = df.rename(columns={"HYP": "prediction"})

    # Defines the minimal column set required by the notebook for loading and evaluation.
    needed = ["language", "file", "prediction"]

    # Validates that all required columns exist after renaming; raises a clear error if any are missing.
    missing = [c for c in needed if c not in df.columns]
    if missing:
        raise ValueError(f"Missing columns {missing}. Your columns are: {list(df.columns)}")

    # Writes a clean CSV containing only the expected columns so later notebook cells can load it without changes.
    df[needed].to_csv(PRED_CSV, index=False)
    print(f"✅ Created {PRED_CSV} from {ALT}")

# Loads the standardized predictions file using the expected filename.
df_pred = pd.read_csv(PRED_CSV)

# Displays a quick preview and the DataFrame shape to confirm the file loaded correctly and has expected rows/columns.
df_pred.head(), df_pred.shape


(  language           file                           prediction
 0       EN    checkin.wav           where is the checking desk
 1       EN    parents.wav                i had lost my parents
 2       EN   suitcase.wav       please i have lost my suitcase
 3       EN  what_time.wav                 what time is my plan
 4       EN      where.wav  where are the restaurants and shops,
 (17, 3))

In [12]:
# Verifies that the expected predictions CSV exists before attempting to load it.
# This prevents silent failures later and provides a clear instruction when the file is missing.
if not os.path.exists(PRED_CSV):
    # Raises a descriptive error explaining the two accepted ways to produce the required CSV:
    # - run the inference step in a compatible environment
    # - generate the CSV externally (e.g., Docker) and copy it into the current folder
    raise FileNotFoundError(
        f"Could not find {PRED_CSV}.\n"
        "Either (1) run the optional inference cell, or (2) generate ex4_predictions.csv in Docker and place it here."
    )

# Loads the predictions CSV into a DataFrame so transcripts can be evaluated and summarized.
df_pred = pd.read_csv(PRED_CSV)

# Displays the first few rows and the full shape to confirm the file loaded correctly and has expected structure.
df_pred.head(), df_pred.shape


(  language           file                           prediction
 0       EN    checkin.wav           where is the checking desk
 1       EN    parents.wav                i had lost my parents
 2       EN   suitcase.wav       please i have lost my suitcase
 3       EN  what_time.wav                 what time is my plan
 4       EN      where.wav  where are the restaurants and shops,
 (17, 3))

## Compute WER per file + summary tables


In [14]:
# Classifies a recording as child or adult using a filename convention.
# Files containing the substring "child" are treated as child speech; all others default to adult.
def speaker_type(fname: str) -> str:
    return "child" if "child" in fname else "adult"

# Infers the language label from the filename suffix.
# Spanish and Italian are identified by the *_es.wav and *_it.wav patterns; all other files default to English.
def language_from_file(fname: str) -> str:
    if fname.endswith("_es.wav"):
        return "Spanish"
    if fname.endswith("_it.wav"):
        return "Italian"
    return "English"

# Accumulates per-file evaluation rows containing reference text, hypothesis text, and computed WER.
rows = []

# Tracks prediction filenames that do not have a corresponding reference transcript entry.
missing_refs = []

# Iterates through each prediction row produced by inference or loaded from the predictions CSV.
for _, r in df_pred.iterrows():
    # Extracts the filename field as a string for dictionary lookup and metadata parsing.
    fname = str(r["file"])

    # Extracts the predicted transcript text as a string for WER computation.
    hyp = str(r["prediction"])

    # Skips evaluation when no reference transcript exists for the given filename.
    # Missing files are collected to make mapping problems visible in the output.
    if fname not in REF:
        missing_refs.append(fname)
        continue

    # Retrieves the ground-truth reference transcript for this file.
    ref = REF[fname]

    # Appends a single evaluation record including derived language/speaker labels and the WER score.
    rows.append({
        "language": language_from_file(fname),
        "speaker": speaker_type(fname),
        "file": fname,
        "ref": ref,
        "hyp": hyp,
        "wer": wer(ref, hyp)
    })

# Builds a DataFrame from the evaluation records and sorts it for a stable, readable report ordering.
df_eval = pd.DataFrame(rows).sort_values(["language", "speaker", "file"]).reset_index(drop=True)

# Prints how many files were successfully evaluated to confirm coverage.
print("Evaluated files:", len(df_eval))

# Prints a warning list when predictions exist without matching reference entries.
# This indicates either a naming mismatch or a missing REF mapping.
if missing_refs:
    print("WARNING: These prediction files have no REF entry:", sorted(set(missing_refs)))

# Displays the evaluation table including WER per file.
df_eval


Evaluated files: 16


,language,speaker,file,ref,hyp,wer
0,English,adult,checkin.wav,Where is the check-in desk?,where is the checking desk,0.333333
1,English,adult,parents.wav,I have lost my parents.,i had lost my parents,0.200000
2,English,adult,suitcase.wav,"Please, I have lost my suitcase.",please i have lost my suitcase,0.000000
3,English,adult,what_time.wav,What time is my plane?,what time is my plan,0.200000
4,English,adult,your_sentence1.wav,I don't want anything to drink i'm rather hungry.,i don't want anything to drink and rather hungry,0.111111
5,English,adult,your_sentence2.wav,I excercise for an hour every morning.,i exercise but an over every morning,0.428571
6,Italian,adult,checkin_it.wav,Dove e' il bancone?,dove e il pancone,0.500000
7,Italian,adult,parents_it.wav,Ho perso i miei genitori.,perso i miei genitori,0.200000
8,Italian,adult,suitcase_it.wav,"Per favore, ho perso la mia valigia.",per fare ho perso la mia valigia,0.142857
9,Italian,adult,what_time_it.wav,A che ora e’ il mio aereo?,nero,1.000000


In [15]:
# Summary: mean WER by language
df_lang = df_eval.groupby("language", as_index=False)["wer"].mean().sort_values("wer")
df_lang

,language,wer
0,English,0.212169
2,Spanish,0.411667
1,Italian,0.454286


In [16]:
# Summary: mean WER by language and speaker type (adult vs child)
df_lang_speaker = df_eval.groupby(["language", "speaker"], as_index=False)["wer"].mean().sort_values(["language", "speaker"])
df_lang_speaker

,language,speaker,wer
0,English,adult,0.212169
1,Italian,adult,0.454286
2,Spanish,adult,0.411667


## Save result

This creates:
- `ex4_wer_per_file.csv`
- `ex4_wer_summary.csv`


In [18]:
# Defines the output CSV path for detailed per-file evaluation results.
# This file contains one row per audio file with language, speaker type, hypothesis, reference, and WER.
out_per_file = os.path.join(BASE_DIR, "deepspeech_results_per_file.csv")

# Defines the output CSV path for aggregated WER statistics.
# This file stores mean WER grouped by language and speaker type for high-level reporting.
out_summary = os.path.join(BASE_DIR, "deepspeech_results_wer_summary.csv")

# Writes the per-file evaluation table to disk for inspection and submission evidence.
df_eval.to_csv(out_per_file, index=False)

# Computes the mean WER for each (language, speaker) group to summarize performance patterns.
summary = df_eval.groupby(["language", "speaker"], as_index=False)["wer"].mean()

# Writes the grouped summary table to disk so the results can be referenced in the report.
summary.to_csv(out_summary, index=False)

# Prints confirmation messages showing exactly where the output files were saved.
print("Saved:", out_per_file)
print("Saved:", out_summary)


Saved: /Users/paris/Desktop/isp/Exercise4/deepspeech/deepspeech_results_per_file.csv
Saved: /Users/paris/Desktop/isp/Exercise4/deepspeech/deepspeech_results_wer_summary.csv


## Analysis  Results

In [20]:
# Fill these automatically from the computed summaries
lang_to_wer = dict(zip(df_lang["language"], df_lang["wer"]))

print(f"- Mean WER (English): {lang_to_wer.get('English', float('nan')):.3f}")
print(f"- Mean WER (Italian): {lang_to_wer.get('Italian', float('nan')):.3f}")
print(f"- Mean WER (Spanish): {lang_to_wer.get('Spanish', float('nan')):.3f}\n")

print("- Higher WER typically indicates mismatch between training data and test conditions (accent, noise, child speech, language model coverage).") 


- Mean WER (English): 0.212
- Mean WER (Italian): 0.454
- Mean WER (Spanish): 0.412

- Higher WER typically indicates mismatch between training data and test conditions (accent, noise, child speech, language model coverage).
